In [1]:
from dotenv import load_dotenv
import os
from anthropic import Anthropic

In [3]:
if os.getenv("ANTHROPIC_API_KEY") is not None:
    print("API key is set")
else:
    print("API key is not set")

API key is set


In [4]:
client = Anthropic()

In [8]:
def add_user_message(messeges, content):
    messeges.append({"role": "user", "content": content})

def add_assistant_message(messeges, content):
    messeges.append({"role": "assistant", "content": content})

def chat_with_claude(messages,stop_sequences=None):
    if stop_sequences is None:
        args={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 1024,
            "messages": messages
        }
    else:
        args={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 1024,
            "messages": messages,
            "stop_sequences": stop_sequences
        }
    response = client.messages.create(**args)
    return response.content[0].text

In [22]:
messages = []

In [23]:
import json

In [24]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    response=chat_with_claude(messages,stop_sequences=["```"])
    return json.loads(response)

In [25]:
json_content = generate_dataset()

In [26]:
json_content

[{'task': 'Write a Python function that takes an AWS S3 bucket name as input and returns True if the bucket name follows AWS naming conventions (lowercase letters, numbers, hyphens only; 3-63 characters; no consecutive periods or hyphens at start/end), False otherwise.'},
 {'task': "Create a JSON object for an AWS IAM policy that allows a user to read objects from a specific S3 bucket named 'my-app-logs' but denies write access to any S3 resources."},
 {'task': 'Write a regex pattern that matches valid AWS EC2 instance IDs (format: i- followed by either 8 or 17 hexadecimal characters, such as i-1234abcd or i-0123456789abcdef0).'}]

In [27]:
with open("evaluation_dataset.json", "w") as f:
    json.dump(json_content, f, indent=4)

In [35]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages=[]
    add_user_message(messages, prompt)
    response=chat_with_claude(messages)
    return response

In [36]:
def run_test_case(test_case):
    """Runs a test case and evaluates the result"""
    response = run_prompt(test_case)
    result={
        "response": response,
        "test_case": test_case,
        "score": 10
    }
    return result

In [37]:
def run_eval(dataset):
    """Runs the evaluation on the dataset and returns the results"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [38]:
with open("evaluation_dataset.json","r") as f:
    dataset = json.load(f)
    results = run_eval(dataset)

In [39]:
results

[{'response': 'Here\'s a Python function that validates AWS S3 bucket names according to the naming conventions:\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name):\n    """\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Can consist only of lowercase letters, numbers, dots (.), and hyphens (-)\n    - Must begin and end with a letter or number\n    - Must not contain two adjacent periods\n    - Must not be formatted as an IP address (e.g., 192.168.5.4)\n    - Must not start with the prefix \'xn--\'\n    - Must not end with the suffix \'-s3alias\'\n    \n    Args:\n        bucket_name (str): The bucket name to validate\n        \n    Returns:\n        bool: True if valid, False otherwise\n    """\n    \n    # Check if input is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Check length (3-63 characters)\n    if len(bucke

In [40]:
with open("evaluation_results.json","w") as f:
    json.dump(results, f, indent=4)

### Add model grader part

In [41]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """.format(task=test_case["task"], solution=output)
    messages=[]
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages,"```json")
    response=chat_with_claude(messages,stop_sequences=["```"])
    return json.loads(response)

In [42]:
def run_test_case(test_case):
    """Runs a test case and evaluates the result"""
    response = run_prompt(test_case)
    eval_result = grade_by_model(test_case, response)
    result={
        "response": response,
        "test_case": test_case,
        "score": eval_result["score"],
        "reasoning": eval_result["reasoning"]
    }
    return result

In [60]:
from statistics import mean


def run_eval(dataset):
    """Runs the evaluation on the dataset and returns the results"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    mean_score = mean([r["score"] for r in results])
    print("Average score: ", mean_score)
    return results

In [61]:
with open("evaluation_dataset.json","r") as f:
    dataset = json.load(f)
    results = run_eval(dataset)

Average score:  7


[6, 8, 7]


6
